In [1]:
# Phase-10 — Domain shift / generalization evaluation
# This notebook:
# 1) evaluates all 3 models source-by-source on val_split.csv
# 2) evaluates all 3 models on SDNET non-cracked images only
# Outputs:
# - phase10_sourcewise_metrics.csv
# - phase10_sourcewise_summary.csv
# - phase10_sdnet_negative_fp_report.csv
# - phase10_sourcewise_barplot.png
# - phase10_sdnet_negative_fp_barplot.png
# - phase10_examples.png

# Cell 1 — Install/check packages

In [2]:
import sys, subprocess

def ensure(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", pkg])

try:
    import segmentation_models_pytorch as smp
except Exception:
    ensure("segmentation-models-pytorch")
    ensure("timm")

try:
    import matplotlib.pyplot as plt
except Exception:
    ensure("matplotlib")
    import matplotlib.pyplot as plt

import torch
import segmentation_models_pytorch as smp

print("torch:", torch.__version__)
print("smp:", smp.__version__)
print("cuda available:", torch.cuda.is_available())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 26.5 MB/s eta 0:00:00
torch: 2.10.0+cpu
smp: 0.5.0
cuda available: False


# Cell 2 — Imports

In [3]:
import os, glob, json, random
from pathlib import Path

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt

from torch.utils.data import Dataset, DataLoader

# Cell 3 — Config

In [4]:
SEED = 7
IMG_SIZE = 448
NUM_WORKERS = 2
BATCH_SIZE = 4

# Risk-controlled thresholds from Phase-8
THRESHOLD_POLICY = {
    "model1_unet": 0.9,
    "model2_deeplabv3plus": 0.5,
    "model3_unetplusplus": 0.5,
}

# To keep runtime manageable on SDNET:
# set to None to use all non-cracked images,
# or use an integer like 1000 / 2000 for quicker testing.
MAX_SDNET_IMAGES_PER_DOMAIN = 1000

OUT_DIR = Path("/kaggle/working/output/phase10")
OUT_DIR.mkdir(parents=True, exist_ok=True)

def seed_everything(seed=7):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

seed_everything(SEED)
print("Threshold policy:", THRESHOLD_POLICY)
print("Output folder:", OUT_DIR)

Threshold policy: {'model1_unet': 0.9, 'model2_deeplabv3plus': 0.5, 'model3_unetplusplus': 0.5}
Output folder: /kaggle/working/output/phase10


# Cell 4 — Device selection

In [5]:
def choose_device():
    if not torch.cuda.is_available():
        print("CUDA not available -> using CPU")
        return torch.device("cpu")

    cap = torch.cuda.get_device_capability(0)
    name = torch.cuda.get_device_name(0)
    print("GPU:", name, "| capability:", cap)

    if cap[0] < 7:
        print(" GPU capability < 7.0 -> using CPU to avoid CUDA crash")
        return torch.device("cpu")

    return torch.device("cuda")

device = choose_device()
print("Using device:", device)

CUDA not available -> using CPU
Using device: cpu


# Cell 5 — Find inputs

In [6]:
VAL_CSV = glob.glob("/kaggle/input/**/val_split.csv", recursive=True)[0]
val_df = pd.read_csv(VAL_CSV)

print("VAL_CSV:", VAL_CSV)
print("val rows:", len(val_df))
print("val columns:", val_df.columns.tolist())
print("unique sources:", sorted(val_df["source"].dropna().unique().tolist()))

def pick_first(patterns):
    found = []
    for pat in patterns:
        found.extend(glob.glob(pat, recursive=True))
    found = sorted(set(found))
    return found[0] if found else None

MODEL1_CKPT = pick_first([
    "/kaggle/input/**/*model-1-checkpoints*/*best*.pt",
    "/kaggle/input/**/*unet_efficientnetb3_best*.pt",
])

MODEL2_CKPT = pick_first([
    "/kaggle/input/**/*model-2-checkpoints*/*best*.pt",
    "/kaggle/input/**/*deeplabv3plus*best*.pt",
])

MODEL3_CKPT = pick_first([
    "/kaggle/input/**/*model-3-checkpoints*/*best*.pt",
    "/kaggle/input/**/*unetplusplus*best*.pt",
])

print("\nMODEL1_CKPT:", MODEL1_CKPT)
print("MODEL2_CKPT:", MODEL2_CKPT)
print("MODEL3_CKPT:", MODEL3_CKPT)

assert MODEL1_CKPT is not None
assert MODEL2_CKPT is not None
assert MODEL3_CKPT is not None

VAL_CSV: /kaggle/input/datasets/arman00p/phase-3-csv/val_split.csv
val rows: 1695
val columns: ['stem', 'image_path', 'mask_path', 'source', 'empty_mask', 'has_crack']
unique sources: ['cfd', 'crack500', 'cracktree200', 'deepcrack', 'eugen', 'forest', 'gaps384', 'noncrack', 'rissbilder', 'sylvie', 'volker']

MODEL1_CKPT: /kaggle/input/datasets/arman00p/model-1-checkpoints-unet-efficientnet-b3/output/SAVED_MODELS/unet_efficientnetb3_best_ep029_20260219_205815.pt
MODEL2_CKPT: /kaggle/input/datasets/arman00p/model-2-checkpointsdeeplabv3plus-efficientnet-b3/deeplabv3plus_efficientnetb3_pt files/deeplabv3plus_efficientnetb3_best_20260222_102303.pt
MODEL3_CKPT: /kaggle/input/datasets/arman00p/model-3-checkpoints-datasetunet-eff-net-b3/unetplusplus_efficientnetb3_best.pt


# Cell 6 — Helper functions

In [7]:
def extract_state_dict_any(ckpt_obj):
    if isinstance(ckpt_obj, dict) and len(ckpt_obj) > 0:
        if all(isinstance(v, torch.Tensor) for v in ckpt_obj.values()):
            return ckpt_obj

    if isinstance(ckpt_obj, dict):
        for key in ["state_dict", "model_state_dict", "model", "net", "weights"]:
            if key in ckpt_obj and isinstance(ckpt_obj[key], dict):
                sd = ckpt_obj[key]
                if len(sd) > 0 and all(isinstance(v, torch.Tensor) for v in sd.values()):
                    return sd

    best = None
    def walk(obj):
        nonlocal best
        if isinstance(obj, dict):
            if len(obj) > 0 and all(isinstance(v, torch.Tensor) for v in obj.values()):
                if best is None or len(obj) > len(best):
                    best = obj
            for v in obj.values():
                walk(v)
        elif isinstance(obj, (list, tuple)):
            for v in obj:
                walk(v)

    walk(ckpt_obj)
    if best is not None:
        return best

    raise ValueError("Could not locate state_dict inside checkpoint")

def strip_prefixes(sd):
    cleaned = {}
    for k, v in sd.items():
        nk = k
        for pref in ("module.", "model.", "net."):
            if nk.startswith(pref):
                nk = nk[len(pref):]
        cleaned[nk] = v
    return cleaned

def iou_empty_aware(gt, pr, eps=1e-7):
    gt = gt.astype(bool)
    pr = pr.astype(bool)
    inter = np.logical_and(gt, pr).sum()
    union = np.logical_or(gt, pr).sum()
    if union == 0:
        return 1.0
    return float(inter / (union + eps))

def dice_empty_aware(gt, pr, eps=1e-7):
    gt = gt.astype(bool)
    pr = pr.astype(bool)
    inter = np.logical_and(gt, pr).sum()
    denom = gt.sum() + pr.sum()
    if denom == 0:
        return 1.0
    return float((2.0 * inter) / (denom + eps))

# Cell 7 — Datasets

In [8]:
class CrackValDataset(Dataset):
    def __init__(self, df: pd.DataFrame, img_size: int, preprocess_fn):
        self.df = df.reset_index(drop=True)
        self.img_size = img_size
        self.preprocess_fn = preprocess_fn

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img = cv2.imread(str(row["image_path"]), cv2.IMREAD_COLOR)
        if img is None:
            raise FileNotFoundError(row["image_path"])
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (self.img_size, self.img_size), interpolation=cv2.INTER_AREA).astype(np.float32)

        m = cv2.imread(str(row["mask_path"]), cv2.IMREAD_GRAYSCALE)
        if m is None:
            raise FileNotFoundError(row["mask_path"])
        m = cv2.resize(m, (self.img_size, self.img_size), interpolation=cv2.INTER_NEAREST)
        m = (m > 127).astype(np.uint8)

        img = self.preprocess_fn(img)

        img_t = torch.from_numpy(img).permute(2, 0, 1).float()
        m_t = torch.from_numpy(m).unsqueeze(0).float()

        meta = {
            "stem": row.get("stem", ""),
            "source": row.get("source", ""),
            "image_path": str(row["image_path"]),
            "mask_path": str(row["mask_path"]),
            "has_crack": int(row.get("has_crack", int(m.sum() > 0))),
        }
        return img_t, m_t, meta

class SDNETNegativeDataset(Dataset):
    def __init__(self, image_paths, img_size, preprocess_fn):
        self.image_paths = image_paths
        self.img_size = img_size
        self.preprocess_fn = preprocess_fn

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        p = self.image_paths[idx]
        img = cv2.imread(str(p), cv2.IMREAD_COLOR)
        if img is None:
            raise FileNotFoundError(p)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (self.img_size, self.img_size), interpolation=cv2.INTER_AREA).astype(np.float32)
        img = self.preprocess_fn(img)
        img_t = torch.from_numpy(img).permute(2, 0, 1).float()
        return img_t, str(p)

# Cell 8 — Load all 3 models

In [9]:
def load_smp_model(arch_name, ckpt_path):
    ckpt = torch.load(ckpt_path, map_location="cpu")
    sd = strip_prefixes(extract_state_dict_any(ckpt))

    encoder_candidates = ["efficientnet-b3", "timm-efficientnet-b3"]
    best = None

    for enc in encoder_candidates:
        if arch_name == "Unet":
            m = smp.Unet(encoder_name=enc, encoder_weights=None, in_channels=3, classes=1, activation=None)
        elif arch_name == "DeepLabV3Plus":
            m = smp.DeepLabV3Plus(encoder_name=enc, encoder_weights=None, in_channels=3, classes=1, activation=None)
        elif arch_name == "UnetPlusPlus":
            m = smp.UnetPlusPlus(encoder_name=enc, encoder_weights=None, in_channels=3, classes=1, activation=None)
        else:
            raise ValueError(f"Unknown arch: {arch_name}")

        missing, unexpected = m.load_state_dict(sd, strict=False)
        score = len(missing) + len(unexpected)
        print(f"[{arch_name} | {enc}] missing={len(missing)} unexpected={len(unexpected)} score={score}")

        if best is None or score < best["score"]:
            best = {"model": m, "encoder": enc, "score": score}

    if best["score"] > 150:
        raise RuntimeError(f"{arch_name} checkpoint load looks wrong. Score={best['score']}")

    return best["model"].to(device).eval(), best["encoder"]

loaded_models = {
    "model1_unet": {"arch": "Unet", "ckpt": MODEL1_CKPT},
    "model2_deeplabv3plus": {"arch": "DeepLabV3Plus", "ckpt": MODEL2_CKPT},
    "model3_unetplusplus": {"arch": "UnetPlusPlus", "ckpt": MODEL3_CKPT},
}

for name, info in loaded_models.items():
    print(f"\nLoading {name} ...")
    model, enc = load_smp_model(info["arch"], info["ckpt"])
    info["model"] = model
    info["encoder"] = enc

print("\n All models loaded")


Loading model1_unet ...
[Unet | efficientnet-b3] missing=494 unexpected=572 score=1066
[Unet | timm-efficientnet-b3] missing=0 unexpected=0 score=0

Loading model2_deeplabv3plus ...
[DeepLabV3Plus | efficientnet-b3] missing=0 unexpected=0 score=0
[DeepLabV3Plus | timm-efficientnet-b3] missing=494 unexpected=572 score=1066

Loading model3_unetplusplus ...
[UnetPlusPlus | efficientnet-b3] missing=0 unexpected=0 score=0
[UnetPlusPlus | timm-efficientnet-b3] missing=494 unexpected=572 score=1066

 All models loaded


# Cell 9 — Source-wise evaluation on val_split.csv

In [10]:
def evaluate_sourcewise(model_name, model, encoder_name, threshold):
    preprocess_fn = smp.encoders.get_preprocessing_fn(encoder_name, pretrained="imagenet")
    ds = CrackValDataset(val_df, img_size=IMG_SIZE, preprocess_fn=preprocess_fn)
    dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

    rows = []

    for imgs, msks, metas in dl:
        imgs = imgs.to(device)
        with torch.inference_mode():
            prob = torch.sigmoid(model(imgs)).cpu().numpy()[:,0]

        gt_batch = msks.numpy()[:,0].astype(np.uint8)

        for b in range(prob.shape[0]):
            gt = gt_batch[b]
            pr = (prob[b] >= threshold).astype(np.uint8)

            rows.append({
                "model_name": model_name,
                "source": metas["source"][b],
                "stem": metas["stem"][b],
                "has_crack_gt": int(gt.sum() > 0),
                "pred_any": int(pr.sum() > 0),
                "iou": iou_empty_aware(gt, pr),
                "dice": dice_empty_aware(gt, pr),
            })

    return pd.DataFrame(rows)

all_source_rows = []
for model_name, info in loaded_models.items():
    print(f"\nEvaluating sourcewise for {model_name} ...")
    df_model = evaluate_sourcewise(
        model_name=model_name,
        model=info["model"],
        encoder_name=info["encoder"],
        threshold=THRESHOLD_POLICY[model_name]
    )
    all_source_rows.append(df_model)

sourcewise_df = pd.concat(all_source_rows, ignore_index=True)
sourcewise_df.to_csv(OUT_DIR / "phase10_sourcewise_metrics.csv", index=False)

summary_rows = []
for (model_name, source), sub in sourcewise_df.groupby(["model_name", "source"]):
    crack = sub[sub["has_crack_gt"] == 1]
    no_crack = sub[sub["has_crack_gt"] == 0]

    summary_rows.append({
        "model_name": model_name,
        "source": source,
        "n_total": len(sub),
        "n_crack": len(crack),
        "n_no_crack": len(no_crack),
        "mean_iou_all": sub["iou"].mean(),
        "mean_iou_crack_only": crack["iou"].mean() if len(crack) else np.nan,
        "mean_dice_crack_only": crack["dice"].mean() if len(crack) else np.nan,
        "no_crack_fp_rate": no_crack["pred_any"].mean() if len(no_crack) else np.nan,
    })

source_summary_df = pd.DataFrame(summary_rows)
source_summary_df.to_csv(OUT_DIR / "phase10_sourcewise_summary.csv", index=False)

print(" Saved sourcewise metrics + summary")
source_summary_df.head()


Evaluating sourcewise for model1_unet ...


config.json:   0%|          | 0.00/106 [00:00<?, ?B/s]


Evaluating sourcewise for model2_deeplabv3plus ...


config.json:   0%|          | 0.00/106 [00:00<?, ?B/s]


Evaluating sourcewise for model3_unetplusplus ...
 Saved sourcewise metrics + summary


,model_name,source,n_total,n_crack,n_no_crack,mean_iou_all,mean_iou_crack_only,mean_dice_crack_only,no_crack_fp_rate
0,model1_unet,cfd,23,23,0,0.368873,0.368873,0.532743,NaN
1,model1_unet,crack500,468,468,0,0.619606,0.619606,0.747438,NaN
2,model1_unet,cracktree200,34,34,0,0.127241,0.127241,0.225372,NaN
3,model1_unet,deepcrack,82,82,0,0.677918,0.677918,0.797550,NaN
4,model1_unet,eugen,12,12,0,0.644205,0.644205,0.780466,NaN


# Cell 10 — Build SDNET negative image list

In [11]:
# Find SDNET non-cracked images
sdnet_noncracked = sorted(glob.glob("/kaggle/input/**/*/Non-cracked/*", recursive=True))
sdnet_noncracked = [p for p in sdnet_noncracked if os.path.isfile(p)]

print("Total SDNET non-cracked files found:", len(sdnet_noncracked))
print("Example:", sdnet_noncracked[:5])

# split by domain
sdnet_domains = {"Decks": [], "Pavements": [], "Walls": []}
for p in sdnet_noncracked:
    pl = p.lower()
    if "/decks/" in pl:
        sdnet_domains["Decks"].append(p)
    elif "/pavements/" in pl:
        sdnet_domains["Pavements"].append(p)
    elif "/walls/" in pl:
        sdnet_domains["Walls"].append(p)

for k, v in sdnet_domains.items():
    if MAX_SDNET_IMAGES_PER_DOMAIN is not None:
        sdnet_domains[k] = v[:MAX_SDNET_IMAGES_PER_DOMAIN]
    print(k, "->", len(sdnet_domains[k]))

Total SDNET non-cracked files found: 47608
Example: ['/kaggle/input/datasets/aniruddhsharma/structural-defects-network-concrete-crack-images/Decks/Non-cracked/7001-1.jpg', '/kaggle/input/datasets/aniruddhsharma/structural-defects-network-concrete-crack-images/Decks/Non-cracked/7001-10.jpg', '/kaggle/input/datasets/aniruddhsharma/structural-defects-network-concrete-crack-images/Decks/Non-cracked/7001-100.jpg', '/kaggle/input/datasets/aniruddhsharma/structural-defects-network-concrete-crack-images/Decks/Non-cracked/7001-101.jpg', '/kaggle/input/datasets/aniruddhsharma/structural-defects-network-concrete-crack-images/Decks/Non-cracked/7001-102.jpg']
Decks -> 1000
Pavements -> 1000
Walls -> 1000


# Cell 11 — SDNET negative-domain FP evaluation

In [12]:
def evaluate_sdnet_negative(model_name, model, encoder_name, threshold):
    preprocess_fn = smp.encoders.get_preprocessing_fn(encoder_name, pretrained="imagenet")
    rows = []

    for domain_name, image_paths in sdnet_domains.items():
        if len(image_paths) == 0:
            continue

        ds = SDNETNegativeDataset(image_paths, img_size=IMG_SIZE, preprocess_fn=preprocess_fn)
        dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

        for imgs, paths in dl:
            imgs = imgs.to(device)
            with torch.inference_mode():
                prob = torch.sigmoid(model(imgs)).cpu().numpy()[:,0]

            for b in range(prob.shape[0]):
                pr = (prob[b] >= threshold).astype(np.uint8)
                rows.append({
                    "model_name": model_name,
                    "sdnet_domain": domain_name,
                    "image_path": paths[b],
                    "pred_any": int(pr.sum() > 0),
                    "pred_pixels": int(pr.sum()),
                })

    return pd.DataFrame(rows)

all_sdnet_rows = []
for model_name, info in loaded_models.items():
    print(f"\nEvaluating SDNET negative-domain FP for {model_name} ...")
    df_model = evaluate_sdnet_negative(
        model_name=model_name,
        model=info["model"],
        encoder_name=info["encoder"],
        threshold=THRESHOLD_POLICY[model_name]
    )
    all_sdnet_rows.append(df_model)

sdnet_df = pd.concat(all_sdnet_rows, ignore_index=True)
sdnet_df.to_csv(OUT_DIR / "phase10_sdnet_negative_fp_per_image.csv", index=False)

sdnet_report = (
    sdnet_df.groupby(["model_name", "sdnet_domain"], as_index=False)
    .agg(
        n_images=("pred_any", "size"),
        fp_rate=("pred_any", "mean"),
        mean_pred_pixels=("pred_pixels", "mean"),
        median_pred_pixels=("pred_pixels", "median"),
    )
)
sdnet_report.to_csv(OUT_DIR / "phase10_sdnet_negative_fp_report.csv", index=False)

print(" Saved SDNET negative FP report")
sdnet_report


Evaluating SDNET negative-domain FP for model1_unet ...

Evaluating SDNET negative-domain FP for model2_deeplabv3plus ...

Evaluating SDNET negative-domain FP for model3_unetplusplus ...
 Saved SDNET negative FP report


,model_name,sdnet_domain,n_images,fp_rate,mean_pred_pixels,median_pred_pixels
0,model1_unet,Decks,1000,0.463,342.873,0.0
1,model1_unet,Pavements,1000,0.566,255.512,19.0
2,model1_unet,Walls,1000,0.355,116.396,0.0
3,model2_deeplabv3plus,Decks,1000,0.347,345.568,0.0
4,model2_deeplabv3plus,Pavements,1000,0.634,262.499,50.0
5,model2_deeplabv3plus,Walls,1000,0.216,184.667,0.0
6,model3_unetplusplus,Decks,1000,0.634,784.324,76.5
7,model3_unetplusplus,Pavements,1000,0.729,1629.956,163.5
8,model3_unetplusplus,Walls,1000,0.516,350.286,8.0


# Cell 12 — Plot source-wise bar chart

In [13]:
plt.figure(figsize=(12,5))

plot_df = source_summary_df.copy()
plot_df["label"] = plot_df["model_name"] + " | " + plot_df["source"].astype(str)

plt.bar(plot_df["label"], plot_df["mean_iou_crack_only"])
plt.xticks(rotation=75, ha="right")
plt.ylabel("Mean IoU (crack-only)")
plt.title("Phase-10 Source-wise Crack-only IoU")
plt.tight_layout()

out_path = OUT_DIR / "phase10_sourcewise_barplot.png"
plt.savefig(out_path, dpi=220)
plt.close()

print(" Saved:", out_path)

 Saved: /kaggle/working/output/phase10/phase10_sourcewise_barplot.png


# Cell 13 — Plot SDNET negative FP bar chart

In [14]:
plt.figure(figsize=(10,5))

plot_df = sdnet_report.copy()
plot_df["label"] = plot_df["model_name"] + " | " + plot_df["sdnet_domain"]

plt.bar(plot_df["label"], plot_df["fp_rate"])
plt.xticks(rotation=45, ha="right")
plt.ylabel("FP rate on SDNET Non-cracked")
plt.title("Phase-10 External Negative-domain FP Rate")
plt.tight_layout()

out_path = OUT_DIR / "phase10_sdnet_negative_fp_barplot.png"
plt.savefig(out_path, dpi=220)
plt.close()

print(" Saved:", out_path)

 Saved: /kaggle/working/output/phase10/phase10_sdnet_negative_fp_barplot.png


# Cell 14 — Example figure (one source example + one SDNET false-positive example if available)

In [15]:
def read_rgb(path):
    img = cv2.imread(path, cv2.IMREAD_COLOR)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
    return img

def read_mask(path):
    m = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    m = cv2.resize(m, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)
    return (m > 127).astype(np.uint8)

def overlay_mask(img, mask, color=(255,0,0), alpha=0.45):
    out = img.copy().astype(np.float32)
    m = mask.astype(bool)
    out[m] = out[m]*(1-alpha) + np.array(color, dtype=np.float32)*alpha
    return out.astype(np.uint8)

# ---- merge paths back into sourcewise_df ----
path_cols = val_df[["stem", "source", "image_path", "mask_path"]].drop_duplicates().copy()
sourcewise_with_paths = sourcewise_df.merge(
    path_cols,
    on=["stem", "source"],
    how="left"
)

# ---- collect examples ----
examples = []

for model_name, info in loaded_models.items():
    # hardest crack-present source sample
    sub = sourcewise_with_paths[
        (sourcewise_with_paths["model_name"] == model_name) &
        (sourcewise_with_paths["has_crack_gt"] == 1)
    ].copy()

    if len(sub):
        worst = sub.sort_values("iou").iloc[0]
        if pd.notna(worst["image_path"]) and pd.notna(worst["mask_path"]):
            examples.append(("source", model_name, worst))

    # one SDNET false positive example if available
    sub_fp = sdnet_df[
        (sdnet_df["model_name"] == model_name) &
        (sdnet_df["pred_any"] == 1)
    ].copy()

    if len(sub_fp):
        fp = sub_fp.iloc[0]
        examples.append(("sdnet", model_name, fp))

# ---- safety check ----
if len(examples) == 0:
    print("No examples found to plot.")
else:
    nrows = len(examples)
    fig, axes = plt.subplots(nrows, 3, figsize=(10, 3*nrows))
    if nrows == 1:
        axes = np.expand_dims(axes, 0)

    r = 0
    for kind, model_name, row in examples:
        info = loaded_models[model_name]
        preprocess_fn = smp.encoders.get_preprocessing_fn(info["encoder"], pretrained="imagenet")

        if kind == "source":
            img = read_rgb(row["image_path"])
            gt = read_mask(row["mask_path"])

            x = torch.from_numpy(preprocess_fn(img.astype(np.float32))).permute(2,0,1).unsqueeze(0).float().to(device)
            with torch.inference_mode():
                prob = torch.sigmoid(info["model"](x)).cpu().numpy()[0,0]
            pr = (prob >= THRESHOLD_POLICY[model_name]).astype(np.uint8)

            panels = [
                (f"{model_name}\nsource sample", img),
                ("GT", gt * 255),
                (f"Pred overlay\nIoU={row['iou']:.3f}", overlay_mask(img, pr)),
            ]

        else:  # sdnet
            img = read_rgb(row["image_path"])

            x = torch.from_numpy(preprocess_fn(img.astype(np.float32))).permute(2,0,1).unsqueeze(0).float().to(device)
            with torch.inference_mode():
                prob = torch.sigmoid(info["model"](x)).cpu().numpy()[0,0]
            pr = (prob >= THRESHOLD_POLICY[model_name]).astype(np.uint8)

            panels = [
                (f"{model_name}\nSDNET non-crack", img),
                ("Pred mask", pr * 255),
                ("Pred overlay", overlay_mask(img, pr)),
            ]

        for c, (title, im) in enumerate(panels):
            ax = axes[r, c]
            ax.imshow(im, cmap="gray" if im.ndim == 2 else None)
            ax.set_title(title, fontsize=9)
            ax.axis("off")

        r += 1

    plt.tight_layout()
    out_path = OUT_DIR / "phase10_examples.png"
    plt.savefig(out_path, dpi=220)
    plt.close()

    print(" Saved:", out_path)

 Saved: /kaggle/working/output/phase10/phase10_examples.png
